# Turn a Folder of PDFs, Configs, and SQL Schemas Into a Queryable Knowledge Graph

A runnable companion to the course project [*Turn a Folder of PDFs, Configs, and SQL Schemas Into a Queryable Knowledge Graph*](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/folder-knowledge-graph). This notebook walks a folder of mixed document types — PDFs (`pypdf`), config files (standard library), and SQL schemas — and builds a `networkx` graph out of the **references** hidden inside them: a schema defines a table, a config value names a table, a PDF mentions a config key. Then it queries the graph.

**No API key, no signup, no network access needed after installing packages.** Every relationship comes from deterministic, hand-written extraction rules — the graph is only as good as its rules, and that honesty is part of the point.

Works the same in Google Colab, Kaggle Notebooks, or Binder.

## Setup: install dependencies

These are the exact packages the local example project (`examples/folder-knowledge-graph/pyproject.toml`) declares.

In [ ]:
!pip install -q pypdf networkx pyvis


## Create a toy sample folder

The real companion example ships a small `data/sample/` folder on disk. Since a fresh Colab/Kaggle/Binder session doesn't have it, we recreate the same files here by writing them directly — this keeps the notebook fully self-contained.

The sample is a tiny **bookstore app** project: three SQL schemas (`users`/`sessions`, `orders`/`order_items`, `books`/`tax_rates`), three config files that name those tables, and three short PDFs that document them.

In [ ]:
from pathlib import Path

ROOT = Path("data/sample")
(ROOT / "sql").mkdir(parents=True, exist_ok=True)
(ROOT / "config").mkdir(parents=True, exist_ok=True)
(ROOT / "pdfs").mkdir(parents=True, exist_ok=True)

(ROOT / "sql/001_users.sql").write_text('''-- Core identity tables: users and their login sessions.
CREATE TABLE users (
    id INTEGER PRIMARY KEY,
    username TEXT NOT NULL UNIQUE,
    email TEXT NOT NULL UNIQUE,
    password_hash TEXT NOT NULL,
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE sessions (
    id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL REFERENCES users(id),
    token TEXT NOT NULL UNIQUE,
    expires_at TEXT NOT NULL
);
''', encoding="utf-8")

(ROOT / "sql/002_orders.sql").write_text('''-- Sales tables: orders belong to a user, order_items point back at a book.
CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL REFERENCES users(id),
    total_cents INTEGER NOT NULL DEFAULT 0,
    status TEXT NOT NULL DEFAULT 'pending'
);

CREATE TABLE order_items (
    id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL REFERENCES orders(id),
    book_id INTEGER NOT NULL REFERENCES books(id),
    quantity INTEGER NOT NULL DEFAULT 1
);
''', encoding="utf-8")

(ROOT / "sql/003_books.sql").write_text('''-- Inventory and pricing tables for the bookstore.
CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    price_cents INTEGER NOT NULL,
    stock INTEGER NOT NULL DEFAULT 0
);

CREATE TABLE tax_rates (
    id INTEGER PRIMARY KEY,
    country_code TEXT NOT NULL UNIQUE,
    rate REAL NOT NULL
);
''', encoding="utf-8")

print("Wrote sql/:", sorted(p.name for p in (ROOT / "sql").glob("*.sql")))


In [ ]:
(ROOT / "config/app.toml").write_text('''# Main application configuration for the bookstore service.

[database]
dbname = "bookstore"
seed_tables = ["users", "books"]
migrate_tables = ["users", "sessions", "orders", "order_items", "books", "tax_rates"]

[auth]
jwt_secret = "change-me-in-prod"
token_ttl_seconds = 3600

[server]
port = 8080
host = "127.0.0.1"
''', encoding="utf-8")

(ROOT / "config/auth.ini").write_text('''; Authentication provider settings.

[auth]
provider = local
session_table = sessions
password_hash_algo = sha256

[database]
connection_pool_size = 5
''', encoding="utf-8")

(ROOT / "config/reporting.toml").write_text('''# Weekly sales report configuration.

[reports.weekly]
enabled = true
source_tables = ["orders", "order_items"]
currency = "USD"

[exports]
include_tax_rates = true
''', encoding="utf-8")

print("Wrote config/:", sorted(p.name for p in (ROOT / "config").glob("*")))


## The tricky part: the sample PDFs

`pypdf` can *read* PDFs, but it doesn't ship a text-layout writer — so the sample PDFs are hand-crafted with a tiny pure-Python PDF writer. A minimal valid PDF is just a handful of objects: a catalog, a page, two fonts, and a content stream that draws text with the classic `BT`/`ET` operators. The companion example's `make_pdf_data.py` does exactly this; here's the same generator inline.

In [ ]:
def _esc(s):
    return s.replace("\\", "\\\\").replace("(", "\\").replace(")", "\\)")

def _wrap(text, width=88):
    lines, cur = [], ""
    for word in text.split():
        candidate = f"{cur} {word}".strip()
        if len(candidate) <= width:
            cur = candidate
        else:
            lines.append(cur); cur = word
    if cur:
        lines.append(cur)
    return lines

def make_pdf(title, paragraphs):
    """Builds a minimal, valid single-page PDF that draws `title` and text."""
    lines = [(title, "F2", 20)]
    for para in paragraphs:
        for chunk in _wrap(para):
            lines.append((chunk, "F1", 11))
        lines.append(("", "F1", 9))
    y, stream = 740, []
    for text, font, size in lines:
        if y < 60:
            break
        if text:
            stream.append(f"BT /{font} {size} Tf 50 {y} Td ({_esc(text)}) Tj ET")
            y -= size + 5
        else:
            y -= 12
    content = "\n".join(stream)
    objects = [
        (1, "<< /Type /Catalog /Pages 2 0 R >>"),
        (2, "<< /Type /Pages /Kids [3 0 R] /Count 1 >>"),
        (3, "<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Resources << /Font << /F1 4 0 R /F2 5 0 R >> >> /Contents 6 0 R >>"),
        (4, "<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>"),
        (5, "<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica-Bold >>"),
        (6, f"<< /Length {len(content)} >>\nstream\n{content}\nendstream"),
    ]
    out = bytearray(b"%PDF-1.4\n%\xe2\xe3\xcf\xd3\n")
    offsets = {}
    for obj_num, body in objects:
        offsets[obj_num] = len(out)
        out += f"{obj_num} 0 obj\n".encode() + body.encode() + b"\nendobj\n"
    xref = len(out)
    out += f"xref\n0 {len(objects)+1}\n".encode() + b"0000000000 65535 f \n"
    for i in range(1, len(objects) + 1):
        out += f"{offsets[i]:010d} 00000 n \n".encode()
    out += f"trailer\n<< /Size {len(objects)+1} /Root 1 0 R >>\nstartxref\n{xref}\n%%EOF\n".encode()
    return bytes(out)


In [ ]:
pdfs = [
    ("architecture.pdf", "System Architecture", [
        "The bookstore app has four layers: web, auth, database, and reporting.",
        "The auth module reads the users and sessions tables, and is configured through auth.ini.",
        "The [database] section of app.toml sets the connection and the seed_tables list.",
        "Orders flow from the web layer into the orders table, and each order_item links back to a book.",
    ]),
    ("onboarding.pdf", "Developer Onboarding", [
        "Welcome to the bookstore codebase. Start by reading the SQL schema in data/sample/sql.",
        "The users table stores login credentials; sessions holds the active tokens.",
        "Run the migrations to create orders, order_items, books, and tax_rates.",
        "Set jwt_secret in auth.ini before the first deploy.",
    ]),
    ("data-model.pdf", "Data Model Overview", [
        "This document defines the database tables used across the service.",
        "users: identity records. sessions: login tokens linked to a user.",
        "orders: purchases placed by a user. order_items: line items referencing a book.",
        "books: catalog entries. tax_rates: the VAT rate per country.",
        "A config key like seed_tables in app.toml controls which tables are pre-filled.",
    ]),
]

for filename, title, paras in pdfs:
    (ROOT / f"pdfs/{filename}").write_bytes(make_pdf(title, paras))

# Sanity check: pypdf must be able to read back the hand-crafted PDFs.
from pypdf import PdfReader

for p in sorted((ROOT / "pdfs").glob("*.pdf")):
    reader = PdfReader(str(p))
    first = (reader.pages[0].extract_text() or "").splitlines()[0]
    print(f"{p.name}: {len(reader.pages)} page(s), first line -> {first!r}")


## Build the graph

This mirrors `build_graph.py` from the companion example almost line-for-line: extract entities per file type (SQL tables + foreign keys, config keys via `tomllib`/`configparser`, PDF text via `pypdf`), then a **second pass** resolves cross-file relationships once every table and config key in the folder is known — the same two-pass shape the codebase version of this project uses for call edges.

In [ ]:
import configparser
import re
import tomllib

import networkx as nx


def _table_node(graph, name):
    node = f"table:{name}"
    if node not in graph:
        graph.add_node(node, kind="table", short_name=name)
    return node


def extract_sql(path, graph, rel):
    current = None
    for line in path.read_text(encoding="utf-8").splitlines():
        create = re.match(r"(?i)^\s*create\s+table\s+([a-z0-9_]+)", line)
        if create:
            current = create.group(1).lower()
            graph.add_edge(rel, _table_node(graph, current), kind="defines")
        ref = re.search(r"(?i)references\s+([a-z0-9_]+)", line)
        if ref and current:
            graph.add_edge(_table_node(graph, current), _table_node(graph, ref.group(1).lower()), kind="references")


def _flatten(data, prefix=""):
    flat = {}
    for key, value in data.items():
        path = f"{prefix}.{key}" if prefix else key
        if isinstance(value, dict):
            flat.update(_flatten(value, path))
        else:
            flat[path] = value
    return flat


def extract_config(path, graph, rel):
    text = path.read_text(encoding="utf-8")
    suffix = path.suffix.lower()
    if suffix == ".toml":
        keys = _flatten(tomllib.loads(text))
    elif suffix in (".ini", ".cfg"):
        parser = configparser.ConfigParser()
        parser.read_string(text)
        keys = {f"{s}.{k}": v for s in parser.sections() for k, v in parser.items(s)}
    else:
        keys = {}
    parsed = []
    for key, value in keys.items():
        node = f"key:{rel}:{key}"
        graph.add_node(node, kind="config_key", short_name=key, file=rel)
        graph.add_edge(rel, node, kind="defines")
        parsed.append((node, f"{key} {value}"))
    return parsed


def _mentions(haystack, name):
    return re.search(rf"\b{re.escape(name)}\b", haystack.lower()) is not None


def build_graph(folder):
    graph = nx.DiGraph()
    config_keys, pdf_files = [], []
    for path in sorted(folder.rglob("*")):
        if not path.is_file() or any(p.startswith(".") for p in path.parts):
            continue
        rel = str(path.relative_to(folder))
        suffix = path.suffix.lower()
        if suffix == ".sql":
            graph.add_node(rel, kind="file", doc_type="sql", short_name=path.name)
            extract_sql(path, graph, rel)
        elif suffix in (".toml", ".ini", ".cfg"):
            graph.add_node(rel, kind="file", doc_type="config", short_name=path.name)
            config_keys.extend(extract_config(path, graph, rel))
        elif suffix == ".pdf":
            text = "\n".join(page.extract_text() or "" for page in PdfReader(str(path)).pages)
            graph.add_node(rel, kind="pdf", short_name=path.name, text=text)
            pdf_files.append(rel)
    # Second pass: resolve cross-file references now that every table/key is known.
    tables = {n.removeprefix("table:"): n for n in graph.nodes if n.startswith("table:")}
    keys_by_name = {}
    for node, data in graph.nodes(data=True):
        if data.get("kind") == "config_key":
            keys_by_name.setdefault(data["short_name"], []).append(node)
            keys_by_name.setdefault(data["short_name"].rsplit(".", 1)[-1], []).append(node)
    for key_node, value_text in config_keys:
        for table_name, table_node in tables.items():
            if _mentions(value_text, table_name):
                graph.add_edge(key_node, table_node, kind="references")
    for rel in pdf_files:
        text = graph.nodes[rel]["text"]
        for table_name, table_node in tables.items():
            if _mentions(text, table_name):
                graph.add_edge(rel, table_node, kind="mentions")
        for key_name, key_nodes in keys_by_name.items():
            if _mentions(text, key_name):
                for key_node in key_nodes:
                    if key_node in graph:
                        graph.add_edge(rel, key_node, kind="mentions")
    return graph


graph = build_graph(ROOT)
print(f"{graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

# Printed adjacency summary: what every document contributed.
for node in sorted(graph.nodes):
    if graph.nodes[node].get("kind") not in {"file", "pdf"}:
        continue
    out = [f"{data.get('kind')} {target}" for _, target, data in graph.out_edges(node, data=True)]
    if out:
        print(f"  {node}: {', '.join(out)}")


## Visualize the graph

`pyvis` wraps the `networkx` graph into a self-contained interactive HTML page — drag nodes, zoom, hover for details. In Google Colab it can be displayed inline with `IPython.display.HTML`; on some platforms it renders blank, in which case the printed adjacency summary above is the reliable fallback.

In [ ]:
_COLORS = {
    "file": "#3b82f6",      # blue -- schema/config container files
    "pdf": "#f59e0b",       # amber -- documents
    "table": "#10b981",     # green -- SQL tables
    "config_key": "#8b5cf6",  # purple -- configuration keys
}
_EDGE_COLORS = {"defines": "#d1d5db", "references": "#ef4444", "mentions": "#f59e0b"}

from pyvis.network import Network

net = Network(height="800px", width="100%", directed=True, notebook=False)
net.barnes_hut()
for node, data in graph.nodes(data=True):
    kind = data.get("kind", "file")
    net.add_node(node, label=data.get("short_name", node), title=f"{kind}: {node}", color=_COLORS.get(kind, "#9ca3af"))
for source, target, data in graph.edges(data=True):
    kind = data.get("kind", "")
    net.add_edge(source, target, title=kind, color=_EDGE_COLORS.get(kind, "#d1d5db"))
net.write_html("graph.html")
print("Wrote graph.html")


In [ ]:
# Colab: display the interactive pyvis HTML inline.
# (On Kaggle/Binder this may render blank -- use the adjacency summary above instead.)
from IPython.display import HTML

HTML(filename="graph.html")


## Query the graph

The payoff: `networkx` traversal answers questions a keyword search over raw files can't — especially *indirect* ones like "which config keys reference table `books`?" where the connection only exists because two separate documents happen to name the same table.

In [ ]:
def configs_for_table(graph, table_name):
    t = f"table:{table_name}"
    if t not in graph:
        return []
    return sorted((src, graph.nodes[src].get("file", "?"))
                  for src, _, data in graph.in_edges(t, data=True)
                  if data.get("kind") == "references" and graph.nodes[src].get("kind") == "config_key")

def entities_mentioning(graph, keyword):
    needle = keyword.lower()
    return sorted(node for node, data in graph.nodes(data=True)
                  if needle in f"{node} {data.get('short_name', '')} {data.get('text', '')}".lower())

def neighbors(graph, node):
    out = sorted(f"{data.get('kind')} -> {t}" for _, t, data in graph.out_edges(node, data=True))
    inc = sorted(f"{s} -> {data.get('kind')}" for s, _, data in graph.in_edges(node, data=True))
    return out, inc

# Q1: which configs reference table 'users'?
print("Q1: which configs reference table 'users'?")
for key_node, file in configs_for_table(graph, "users"):
    print(f"  {key_node}  (in {file})")

print()

# Q2: list all entities that mention 'auth'
print("Q2: list all entities that mention 'auth'")
for node in entities_mentioning(graph, "auth"):
    print(f"  {node}")

print()

# Indirect: what touches table:books, and where did those edges come from?
out, inc = neighbors(graph, "table:books")
print("Neighbors of table:books")
for item in out:
    print(f"  {item}")
for item in inc:
    print(f"  {item}")


## Next steps

- Point `build_graph` at your own mixed folder of PDFs/configs/SQL and see what relationships it finds that you didn't know were there.
- Try graph metrics on top: `nx.pagerank(graph)` or in-degree centrality to find the most-referenced tables or config keys.
- The honest upgrade this project deliberately skips: an LLM doing the relation extraction, so relationships the hand-written rules miss (synonyms, prose like "the login table", cross-document concepts) start showing up — see the lesson's [Next steps](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/folder-knowledge-graph) for how.
- See the full lesson at [`docs/projects/folder-knowledge-graph/index.md`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/folder-knowledge-graph) and the local, `uv`-based companion script at [`examples/folder-knowledge-graph/build_graph.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/folder-knowledge-graph).